# Multi-board job scraping test\n\nA notebook-style, **local** test harness inspired by [JobSentinel](https://github.com/cboyd0319/JobSentinel).\n\nThis notebook **prefers public endpoints** when they exist:\n- **Greenhouse**: public boards API\n- **Lever**: public postings API\n- **RemoteOK / WeWorkRemotely / Wellfound**: RSS feeds\n- **HN Who's Hiring**: pulls the latest thread via Algolia API\n- **YC hiring companies**: `yc-oss/api` (companies that are hiring)\n\nFor boards JobSpy supports (**Indeed, LinkedIn, ZipRecruiter, Google, Glassdoor**), you can use `python-jobspy` through the UI below.\n\n*Memory-conscious: small defaults, row limits, normalized columns.*

## 1. Imports

In [ ]:
from __future__ import annotations\n\nimport json\nimport re\nfrom dataclasses import dataclass\nfrom datetime import datetime, timezone\nfrom typing import Any, Dict, Iterable, List, Optional\n\nimport feedparser\nimport pandas as pd\nimport requests\nimport ipywidgets as w\nfrom IPython.display import display as ipy_display\n\npd.set_option("display.max_colwidth", 120)\n\ndef _now_iso() -> str:\n    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")\n\ndef _safe_get(d: Any, *keys: str, default: Any = "") -> Any:\n    cur = d\n    for k in keys:\n        if isinstance(cur, dict) and k in cur:\n            cur = cur[k]\n        else:\n            return default\n    return cur\n\ndef _norm_job(title: str, company: str, location: str, url: str, source: str, posted_at: str = "", extras: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:\n    return {\n        "title": (title or "").strip(),\n        "company": (company or "").strip(),\n        "location": (location or "").strip(),\n        "url": (url or "").strip(),\n        "source": source,\n        "posted_at": posted_at,\n        "fetched_at": _now_iso(),\n        **(extras or {}),\n    }\n\nSESSION = requests.Session()\nSESSION.headers.update({\n    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) JobNotebook/1.0"\n})

## 2. Source adapters (public endpoints)

In [ ]:
def scrape_greenhouse(company_slug: str, query: str = "", limit: int = 50) -> List[Dict[str, Any]]:\n    company_slug = (company_slug or "").strip()\n    if not company_slug:\n        return []\n    url = f"https://boards-api.greenhouse.io/v1/boards/{company_slug}/jobs"\n    r = SESSION.get(url, timeout=20)\n    r.raise_for_status()\n    data = r.json()\n    jobs = []\n    q = (query or "").strip().lower()\n    for j in data.get("jobs", [])[: max(1000, limit * 5)]:\n        title = j.get("title", "")\n        if q and q not in title.lower():\n            continue\n        loc = _safe_get(j, "location", "name", default="")\n        jobs.append(_norm_job(title=title, company=company_slug, location=loc, url=j.get("absolute_url", ""), source="greenhouse"))\n        if len(jobs) >= limit:\n            break\n    return jobs\n\ndef scrape_lever(company_slug: str, query: str = "", limit: int = 50) -> List[Dict[str, Any]]:\n    company_slug = (company_slug or "").strip()\n    if not company_slug:\n        return []\n    url = f"https://api.lever.co/v0/postings/{company_slug}?mode=json"\n    r = SESSION.get(url, timeout=20)\n    r.raise_for_status()\n    data = r.json()\n    jobs = []\n    q = (query or "").strip().lower()\n    for j in data[: max(1000, limit * 5)]:\n        title = j.get("text", "")\n        if q and q not in title.lower():\n            continue\n        loc = j.get("categories", {}).get("location", "") if isinstance(j.get("categories"), dict) else ""\n        jobs.append(_norm_job(title=title, company=company_slug, location=loc, url=j.get("hostedUrl", ""), source="lever"))\n        if len(jobs) >= limit:\n            break\n    return jobs\n\ndef scrape_rss(url: str, source: str, query: str = "", limit: int = 50) -> List[Dict[str, Any]]:\n    d = feedparser.parse(url)\n    jobs: List[Dict[str, Any]] = []\n    q = (query or "").strip().lower()\n    for e in d.entries:\n        title = getattr(e, "title", "")\n        summary = getattr(e, "summary", getattr(e, "description", ""))\n        if q and (q not in title.lower()) and (q not in summary.lower()):\n            continue\n        link = getattr(e, "link", "")\n        published = getattr(e, "published", getattr(e, "updated", ""))\n        jobs.append(_norm_job(title=title, company="", location="", url=link, source=source, posted_at=str(published)))\n        if len(jobs) >= limit:\n            break\n    return jobs\n\ndef scrape_hn_whos_hiring(query: str = "", limit: int = 50) -> List[Dict[str, Any]]:\n    # Find latest 'Ask HN: Who is hiring?' story\n    s = SESSION.get("https://hn.algolia.com/api/v1/search_by_date?tags=story&query=Ask%20HN:%20Who%20is%20hiring", timeout=20).json()\n    hits = s.get("hits", [])\n    if not hits:\n        return []\n    story_id = hits[0].get("objectID")\n    item = SESSION.get(f"https://hn.algolia.com/api/v1/items/{story_id}", timeout=20).json()\n    q = (query or "").strip().lower()\n    out: List[Dict[str, Any]] = []\n    for c in item.get("children", []):\n        text = (c.get("text") or "")\n        if not text:\n            continue\n        plain = re.sub(r"<[^>]+>", " ", text)\n        if q and q not in plain.lower():\n            continue\n        url = f"https://news.ycombinator.com/item?id={c.get('id')}"\n        out.append(_norm_job(title=plain[:120].strip().replace("\n", " ") + "...", company="", location="", url=url, source="hn_whos_hiring"))\n        if len(out) >= limit:\n            break\n    return out\n\ndef scrape_yc_hiring_companies(query: str = "", limit: int = 50) -> List[Dict[str, Any]]:\n    data = SESSION.get("https://yc-oss.github.io/api/companies/hiring.json", timeout=25).json()\n    q = (query or "").strip().lower()\n    out: List[Dict[str, Any]] = []\n    for c in data:\n        name = c.get("name", "")\n        one = (name + " " + (c.get("one_liner") or "") + " " + (c.get("industry") or "")).lower()\n        if q and q not in one:\n            continue\n        url = c.get("website", "") or c.get("url", "") or ""\n        loc = c.get("all_locations", "") or c.get("location", "") or ""\n        out.append(_norm_job(title=name, company=name, location=str(loc), url=str(url), source="yc_hiring_companies", extras={"one_liner": c.get("one_liner", "")}))\n        if len(out) >= limit:\n            break\n    return out

## 3. UI: choose boards + run\n\nBoards covered here:\n- **Greenhouse** (requires company slug)\n- **Lever** (requires company slug)\n- **RemoteOK / WeWorkRemotely / Wellfound** (RSS feeds)\n- **HN Who's Hiring** (Algolia API)\n- **YC hiring companies** (`yc-oss/api`)\n- **JobSpy boards**: Indeed, LinkedIn, ZipRecruiter, Google, Glassdoor\n\nNot implemented in this notebook (usually ToS/anti-bot heavy or no stable public feed): BuiltIn, Dice, YC Startup Jobs (job listings), JobsWithGPT.\nIf you find an official RSS/JSON endpoint for these later, we can add adapters.

In [ ]:
BOARD_OPTIONS = [\n    ("Greenhouse (API)", "greenhouse"),\n    ("Lever (API)", "lever"),\n    ("RemoteOK (RSS)", "remoteok"),\n    ("WeWorkRemotely (RSS)", "weworkremotely"),\n    ("Wellfound (RSS)", "wellfound"),\n    ("HN Who's Hiring (Algolia)", "hn_whos_hiring"),\n    ("YC hiring companies (yc-oss/api)", "yc_hiring_companies"),\n    ("JobSpy (Indeed/LinkedIn/ZipRecruiter/Google/Glassdoor)", "jobspy"),\n]\n\nboards = w.SelectMultiple(options=BOARD_OPTIONS, value=("remoteok", "weworkremotely"), description="Boards", rows=6)\nquery = w.Text(value="software engineer", description="Query", style={"description_width": "60px"})\nlimit = w.IntSlider(value=25, min=5, max=200, step=5, description="Limit")\n\ngreenhouse_slug = w.Text(value="", placeholder="e.g. stripe", description="GH slug", style={"description_width": "60px"})\nlever_slug = w.Text(value="", placeholder="e.g. netflix", description="Lever slug", style={"description_width": "60px"})\n\njobspy_sites = w.SelectMultiple(\n    options=[("Indeed", "indeed"), ("LinkedIn", "linkedin"), ("ZipRecruiter", "zip_recruiter"), ("Google", "google"), ("Glassdoor", "glassdoor")],\n    value=("indeed",),\n    description="JobSpy",\n    rows=3,\n)\njobspy_location = w.Text(value="Remote", description="Location", style={"description_width": "60px"})\njobspy_country = w.Dropdown(options=["USA", "UK", "Canada", "India", "Australia", "Germany", "France"], value="USA", description="Country")\njobspy_hours_old = w.Dropdown(options=[("1 day", 24), ("3 days", 72), ("7 days", 168), ("14 days", 336)], value=168, description="Hours")\njobspy_is_remote = w.Checkbox(value=True, description="Remote")\n\nout = w.Output()\n\ndef run(_):\n    with out:\n        out.clear_output()\n        q = query.value.strip()\n        n = int(limit.value)\n        chosen = list(boards.value)\n        all_jobs: List[Dict[str, Any]] = []\n        \n        if "greenhouse" in chosen:\n            gh = scrape_greenhouse(greenhouse_slug.value, query=q, limit=n)\n            print(f"Greenhouse: {len(gh)}")\n            all_jobs.extend(gh)\n        if "lever" in chosen:\n            lv = scrape_lever(lever_slug.value, query=q, limit=n)\n            print(f"Lever: {len(lv)}")\n            all_jobs.extend(lv)\n        if "remoteok" in chosen:\n            ro = scrape_rss("https://remoteok.com/remote-jobs.rss", "remoteok", query=q, limit=n)\n            print(f"RemoteOK: {len(ro)}")\n            all_jobs.extend(ro)\n        if "weworkremotely" in chosen:\n            wwr = scrape_rss("https://weworkremotely.com/remote-jobs.rss", "weworkremotely", query=q, limit=n)\n            print(f"WeWorkRemotely: {len(wwr)}")\n            all_jobs.extend(wwr)\n        if "wellfound" in chosen:\n            # Wellfound RSS supports keywords + remote=true.\n            kw = requests.utils.quote(q or "software")\n            wf_url = f"https://wellfound.com/jobs.rss?keywords={kw}&remote=true"\n            wf = scrape_rss(wf_url, "wellfound", query="", limit=n)\n            print(f"Wellfound: {len(wf)}")\n            all_jobs.extend(wf)\n        if "hn_whos_hiring" in chosen:\n            hn = scrape_hn_whos_hiring(query=q, limit=n)\n            print(f"HN Who's Hiring: {len(hn)}")\n            all_jobs.extend(hn)\n        if "yc_hiring_companies" in chosen:\n            yc = scrape_yc_hiring_companies(query=q, limit=n)\n            print(f"YC hiring companies: {len(yc)}")\n            all_jobs.extend(yc)\n        if "jobspy" in chosen:\n            try:\n                from jobspy import scrape_jobs\n                df = scrape_jobs(\n                    site_name=list(jobspy_sites.value) or ["indeed"],\n                    search_term=q or "software engineer",\n                    location=jobspy_location.value.strip(),\n                    results_wanted=min(100, n),\n                    hours_old=int(jobspy_hours_old.value),\n                    country_indeed=jobspy_country.value,\n                    is_remote=bool(jobspy_is_remote.value),\n                    verbose=0,\n                )\n                recs = df.to_dict(orient="records") if df is not None else []\n                print(f"JobSpy: {len(recs)}")\n                for r in recs[:n]:\n                    all_jobs.append(_norm_job(\n                        title=r.get("title", ""),\n                        company=r.get("company", ""),\n                        location=r.get("location", ""),\n                        url=r.get("job_url", ""),\n                        source=f"jobspy_{r.get('site','')}",\n                        posted_at=str(r.get("date_posted", "")),\n                    ))\n            except Exception as e:\n                print("JobSpy failed:", e)\n        \n        if not all_jobs:\n            print("No results. For Greenhouse/Lever, add a company slug. For JobSpy, pick at least one site.")\n            return\n        \n        # De-dupe by URL, keep first\n        seen = set()\n        deduped = []\n        for j in all_jobs:\n            u = (j.get("url") or "").strip()\n            if u and u in seen:\n                continue\n            if u:\n                seen.add(u)\n            deduped.append(j)\n        \n        global jobs_df\n        jobs_df = pd.DataFrame(deduped)\n        cols = [c for c in ["source", "title", "company", "location", "posted_at", "url"] if c in jobs_df.columns]\n        print(f"Total: {len(jobs_df)} (deduped)")\n        ipy_display(jobs_df[cols].head(50) if cols else jobs_df.head(50))\n\nrun_btn = w.Button(description="Run", button_style="primary")\nrun_btn.on_click(run)\n\nform = w.VBox([\n    boards,\n    query,\n    limit,\n    w.HBox([greenhouse_slug, lever_slug]),\n    w.HTML("<b>JobSpy options (only used when JobSpy is selected)</b>"),\n    w.HBox([jobspy_sites, w.VBox([jobspy_location, jobspy_country, jobspy_hours_old, jobspy_is_remote])]),\n    run_btn,\n    out,\n])\nipy_display(form)

## 4. Export (optional)

In [ ]:
import csv\n\ntry:\n    jobs_df\nexcept NameError:\n    print("jobs_df not defined yet. Run the UI cell first.")\nelse:\n    out_path = "jobs_multi.csv"\n    jobs_df.to_csv(out_path, quoting=csv.QUOTE_NONNUMERIC, escapechar="\\", index=False)\n    print(f"Saved: {out_path}")